In [1]:
%env RANK=0
%env WORLD_SIZE=1
%env MASTER_ADDR=127.0.0.1
%env MASTER_PORT=2020

env: RANK=0
env: WORLD_SIZE=1
env: MASTER_ADDR=127.0.0.1
env: MASTER_PORT=2020


In [2]:
llama_checkpoint_dir = "modified_llama/llama-2-7b"
tokenizer_path = "modified_llama/tokenizer.model"
compression_checkpoint_file = "../rigel-data/hierarchical-compression-checkpoint/attention_model.pt"
cv_db_dir = "../rigel-data/context-vectors-compressed"
max_seq_len = 1024
max_batch_size = 4

In [3]:
from modified_llama.llama import Llama
import torch

# Create the Llama generator
print("Building generator...")
generator = Llama.build(
    ckpt_dir=llama_checkpoint_dir,
    tokenizer_path=tokenizer_path,
    max_seq_len=max_seq_len,
    max_batch_size=max_batch_size,
)
print("Built generator!")

Building generator...
> initializing model parallel with size 1
> initializing ddp with size 1
> initializing pipeline with size 1


/home/ubuntu/rigel/modified_llama/llama/generation.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location="cpu")
/home/ubuntu/

Loaded in 14.64 seconds
Built generator!


In [4]:
from cv_library.compressor import Compressor
from cv_library.loss_functions import sequence_similarity
from cv_hier_storage import ContextVectorHierDB, DBConfig

from pathlib import Path

config = DBConfig([16, 64, 256, 1024, 4096])
cv_db = ContextVectorHierDB(Path(cv_db_dir), config, sequence_similarity)

torch.set_default_dtype(torch.float32)
compressor = Compressor(compression_checkpoint_file)

Layer #1 output size: 1024
Layer #2 output size: 256
Layer #3 output size: 64
Layer #4 output size: 16


In [5]:
query_string = "Tell me about the Roman god of the sea"

In [6]:
# Tokenize the content and query, and generate context vectors for them
query_tokens = generator.tokenize(max_seq_len, [("", query_string)])
query_tokens = [l for _, l in query_tokens]

_, query_cvs = generator.generate_context_vectors(query_tokens, len(query_tokens[0]))

In [7]:
import torch

with torch.no_grad():
    query_cv = query_cvs[16].to(torch.float32)
    compressed_cvs = compressor.compress(query_cv)
    compressed_cvs = [query_cv, *compressed_cvs]
    compressed_cvs.reverse()
    results = cv_db.search(compressed_cvs, 16)

In [10]:
(output, _) = generator.generate(query_tokens, results[0].cv.to(torch.float16).unsqueeze(0), 16, 1024)

In [11]:
generator.tokenizer.decode(output[0])

'of the\n everybody - and T\n  T a sc - T 1 a -  of T -\n  and T T the the\n  T a T the T - a and V1 the.  \n  \n - - a \n  f the -. T \n the  T M - \n\n T -0 a -\n  d  T  -   the hair 1 T   -.\n the\n d   the the  of  0 M\n  the f - T - \n the  the  the.1  - the T T T . \n  M - the  M1\n - a T -  a111 \n  \n -\n  f - T \n\n the. 1 the  T -\n \n T - T the - the\n T - T - the M T\n The the - st the T\n of T.\n n T - - T \n\n T \n - Y - - 1 st T M -   Sand  - the the Sand the\n10 of  the the -  T.0 the  the  T   a  and  n   the   - the  the T the E T and T00   the  and a T the T 11 T M the the0 T the -1  T T  hair - \n T and - - T a T\n the  -  a\n  T -  - \n the the  - the T the\n - T -  - \n  T the T T  data T. - - a - the -0\n T 1 - the - the and the T\n hair 0 a The T  - the\n T0 the   E T the M  hair T  T1 -  the T the M - - n T of -    T  Sand  T  st the  the. -  -. - T - of  the. the the the T1   E - -. of. T st   -0 0 1 n  T T  a T - 0  T  the T the T  T and T  T  T0  E1  v  T   